In [ ]:
import random
import torch
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torchvision.models as models
import torch.nn.functional as F
from torchvision.models import EfficientNet_B3_Weights
import torch.nn as nn
import torch.optim as optim
import numpy as np
import os
from torch.utils.data import Dataset
from PIL import Image
import pandas as pd

In [ ]:
batch = 32
size = (300, 300)
num_epochs = 30
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="PIL.Image")
warnings.filterwarnings("ignore", message="Corrupt EXIF data.*", category=UserWarning)

In [ ]:
train_transforms = transforms.Compose([
    transforms.Lambda(lambda img: img.convert('RGB')),
    transforms.RandomResizedCrop(
        300,
        scale=(0.65, 1.0),
        ratio=(0.75, 1.33)
    ),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.15,
        hue=0.03
    ),
    transforms.ToTensor(),                     
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],         
        std=[0.229, 0.224, 0.225]
    )
])

val_transforms = transforms.Compose([
    transforms.Lambda(lambda img: img.convert('RGB')),
    transforms.Resize(320),
    transforms.CenterCrop(300),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
def build_hierarchy_mappings(label_to_id_path):
    label_df = pd.read_csv(label_to_id_path)

    label_df = label_df.sort_values("label_id").reset_index(drop=True)

    label_df[["class_name", "subclass_name"]] = label_df["label_name"].str.split(
        "/", n=1, expand=True
    )

    class_names = label_df["class_name"].drop_duplicates().tolist()

    class_to_idx = {
        class_name: idx
        for idx, class_name in enumerate(class_names)
    }

    idx_to_class = {
        idx: class_name
        for class_name, idx in class_to_idx.items()
    }

    subclass_to_local_idx = {}
    local_idx_to_subclass = {}
    num_subclasses_per_class = []

    for class_name in class_names:
        class_part = label_df[label_df["class_name"] == class_name]
        class_part = class_part.sort_values("label_id")

        subclass_names = class_part["subclass_name"].tolist()

        subclass_to_local_idx[class_name] = {
            subclass_name: idx
            for idx, subclass_name in enumerate(subclass_names)
        }

        class_idx = class_to_idx[class_name]

        local_idx_to_subclass[class_idx] = {
            idx: subclass_name
            for idx, subclass_name in enumerate(subclass_names)
        }

        num_subclasses_per_class.append(len(subclass_names))

    return {
        "class_to_idx": class_to_idx,
        "idx_to_class": idx_to_class,
        "subclass_to_local_idx": subclass_to_local_idx,
        "local_idx_to_subclass": local_idx_to_subclass,
        "num_subclasses_per_class": num_subclasses_per_class,
        "label_df": label_df
    }

In [ ]:
mappings = build_hierarchy_mappings("label_to_id.csv")

class_to_idx = mappings["class_to_idx"]
subclass_to_local_idx = mappings["subclass_to_local_idx"]
num_subclasses_per_class = mappings["num_subclasses_per_class"]

print(class_to_idx)
print(subclass_to_local_idx)
print(num_subclasses_per_class)

In [ ]:
import json

save_mappings = {
    "class_to_idx": class_to_idx,
    "idx_to_class": mappings["idx_to_class"],
    "subclass_to_local_idx": subclass_to_local_idx,
    "local_idx_to_subclass": mappings["local_idx_to_subclass"],
    "num_subclasses_per_class": num_subclasses_per_class
}

with open("hierarchy_mappings.json", "w", encoding="utf-8") as f:
    json.dump(save_mappings, f, ensure_ascii=False, indent=4)

In [ ]:
class CustomDataset(Dataset):
    def __init__(
        self,
        csv_path,
        img_dir,
        class_to_idx,
        subclass_to_local_idx,
        transform=None
    ):
        self.df = pd.read_csv(csv_path, sep='\t')
        self.img_dir = img_dir
        self.class_to_idx = class_to_idx
        self.subclass_to_local_idx = subclass_to_local_idx
        self.transform = transform
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_name = row["image_name"]
        class_name = row["class"]
        subclass_name = row["subclass"]

        image_path = os.path.join(self.img_dir, image_name)

        image = Image.open(image_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        class_idx = self.class_to_idx[class_name]

        subclass_local_idx = self.subclass_to_local_idx[class_name][subclass_name]

        return image, class_idx, subclass_local_idx

In [ ]:
annotation_path = "train_with_classes.csv"
train_dataset = CustomDataset(
    csv_path=annotation_path,
    img_dir='data/train',
    transform=train_transforms
)

val_dataset = CustomDataset(
    csv_path=annotation_path,
    img_dir='data/val',
    transform=val_transforms
)

test_dataset = CustomDataset(
    csv_path=annotation_path,
    img_dir='data/test',
    transform=val_transforms
)

train_loader = DataLoader(train_dataset, batch_size=batch, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

In [ ]:
class HierarchicalEfficientNetB3(nn.Module):
    def __init__(self, num_classes, num_subclasses_per_class, dropout=0.3, weights=EfficientNet_B3_Weights.DEFAULT):
        super().__init__()

        self.backbone = models.efficientnet_b3(weights=weights)

        in_features = self.backbone.classifier[1].in_features

        self.backbone.classifier = nn.Identity()

        self.class_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_features, num_classes)
        )
        
        self.subclass_heads = nn.ModuleList([
            nn.Sequential(
                nn.Dropout(dropout),
                nn.Linear(in_features, num_subclasses)
            )
            for num_subclasses in num_subclasses_per_class
        ])
        self._freeze_backbone()
    
    def _freeze_backbone(self):
        for param in self.backbone.parameters():
            param.requires_grad = False

        for block in self.backbone.features[-3:]:
            for param in block.parameters():
                param.requires_grad = True

        for param in self.class_head.parameters():
            param.requires_grad = True

        for head in self.subclass_heads:
            for param in head.parameters():
                param.requires_grad = True
    
    def forward(self, x):
        features = self.backbone(x)

        class_logits = self.class_head(features)
        
        subclass_logits_list = [
            head(features)
            for head in self.subclass_heads
        ]
        
        return class_logits, subclass_logits_list

In [ ]:
def hierarchical_loss(
    class_logits,
    subclass_logits_list,
    class_labels,
    subclass_local_labels,
    class_weight=1.0,
    subclass_weight=1.0,
    subclass_weights_per_class=None
):
    class_loss = F.cross_entropy(class_logits, class_labels)

    subclass_loss = 0.0
    used_samples = 0

    for class_idx, subclass_logits in enumerate(subclass_logits_list):
        mask = class_labels == class_idx

        if mask.any():
            weights = None
            if subclass_weights_per_class is not None:
                weights = subclass_weights_per_class[class_idx].to(subclass_logits.device)

            loss = F.cross_entropy(
                subclass_logits[mask],
                subclass_local_labels[mask],
                weight=weights,
                reduction="sum"
            )

            subclass_loss += loss
            used_samples += mask.sum().item()

    subclass_loss = subclass_loss / used_samples

    total_loss = class_weight * class_loss + subclass_weight * subclass_loss

    return total_loss, class_loss, subclass_loss

In [ ]:
num_classes = 6
subclass_count = [8, 5, 9, 10, 5, 3]

In [ ]:

model = HierarchicalEfficientNetB3(6, subclass_count).to(device)

In [ ]:
from tqdm import tqdm

NUM_EPOCHS = 30
WARMUP_EPOCHS = 3

CLASS_WEIGHT = 1.0
SUBCLASS_WEIGHT = 1.0

BACKBONE_LR = 3e-5
HEAD_LR = 1e-3
WEIGHT_DECAY = 1e-4

BEST_MODEL_PATH = "EfficientNetB3_hierarchical_best.pth"
LAST_MODEL_PATH = "EfficientNetB3_hierarchical_last.pth"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

use_amp = device.type == "cuda"
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

In [ ]:
def set_trainable_heads_only(model):
    for param in model.backbone.parameters():
        param.requires_grad = False

    for param in model.class_head.parameters():
        param.requires_grad = True

    for head in model.subclass_heads:
        for param in head.parameters():
            param.requires_grad = True


def set_trainable_last_blocks(model, n_last_blocks=3):
    for param in model.backbone.parameters():
        param.requires_grad = False

    for block in model.backbone.features[-n_last_blocks:]:
        for param in block.parameters():
            param.requires_grad = True

    for param in model.class_head.parameters():
        param.requires_grad = True

    for head in model.subclass_heads:
        for param in head.parameters():
            param.requires_grad = True

In [ ]:
def build_optimizer(model):
    backbone_params = [
        param for param in model.backbone.parameters()
        if param.requires_grad
    ]

    optimizer = torch.optim.AdamW(
        [
            {
                "params": backbone_params,
                "lr": BACKBONE_LR
            },
            {
                "params": model.class_head.parameters(),
                "lr": HEAD_LR
            },
            {
                "params": model.subclass_heads.parameters(),
                "lr": HEAD_LR
            }
        ],
        weight_decay=WEIGHT_DECAY
    )

    return optimizer

In [ ]:
def calculate_batch_metrics(
    class_logits,
    subclass_logits_list,
    class_labels,
    subclass_labels
):
    batch_size = class_labels.size(0)

    class_preds = class_logits.argmax(dim=1)
    class_correct = (class_preds == class_labels).sum().item()

    subclass_correct_true_class = 0

    for class_idx, subclass_logits in enumerate(subclass_logits_list):
        mask = class_labels == class_idx

        if mask.any():
            subclass_preds = subclass_logits[mask].argmax(dim=1)
            subclass_correct_true_class += (
                subclass_preds == subclass_labels[mask]
            ).sum().item()

    joint_correct = 0

    for i in range(batch_size):
        pred_class = class_preds[i].item()
        true_class = class_labels[i].item()

        if pred_class != true_class:
            continue

        pred_subclass = subclass_logits_list[pred_class][i].argmax().item()
        true_subclass = subclass_labels[i].item()

        if pred_subclass == true_subclass:
            joint_correct += 1

    return {
        "class_correct": class_correct,
        "subclass_correct_true_class": subclass_correct_true_class,
        "joint_correct": joint_correct,
        "total": batch_size
    }

In [ ]:
def train_one_epoch(
    model,
    train_loader,
    optimizer,
    device,
    scaler,
    use_amp=True
):
    model.train()

    total_loss_sum = 0.0
    class_loss_sum = 0.0
    subclass_loss_sum = 0.0

    class_correct = 0
    subclass_correct_true_class = 0
    joint_correct = 0
    total_samples = 0

    progress_bar = tqdm(train_loader, desc="Train", leave=False)

    for images, class_labels, subclass_labels in progress_bar:
        images = images.to(device)
        class_labels = class_labels.to(device).long()
        subclass_labels = subclass_labels.to(device).long()

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=use_amp):
            class_logits, subclass_logits_list = model(images)

            total_loss, class_loss, subclass_loss = hierarchical_loss(
                class_logits=class_logits,
                subclass_logits_list=subclass_logits_list,
                class_labels=class_labels,
                subclass_labels=subclass_labels,
                class_weight=CLASS_WEIGHT,
                subclass_weight=SUBCLASS_WEIGHT
            )

        scaler.scale(total_loss).backward()
        scaler.step(optimizer)
        scaler.update()

        batch_size = images.size(0)

        total_loss_sum += total_loss.item() * batch_size
        class_loss_sum += class_loss.item() * batch_size
        subclass_loss_sum += subclass_loss.item() * batch_size

        metrics = calculate_batch_metrics(
            class_logits=class_logits.detach(),
            subclass_logits_list=[x.detach() for x in subclass_logits_list],
            class_labels=class_labels,
            subclass_labels=subclass_labels
        )

        class_correct += metrics["class_correct"]
        subclass_correct_true_class += metrics["subclass_correct_true_class"]
        joint_correct += metrics["joint_correct"]
        total_samples += metrics["total"]

        progress_bar.set_postfix({
            "loss": total_loss_sum / total_samples,
            "class_acc": class_correct / total_samples,
            "sub_acc": subclass_correct_true_class / total_samples,
            "joint_acc": joint_correct / total_samples
        })

    return {
        "loss": total_loss_sum / total_samples,
        "class_loss": class_loss_sum / total_samples,
        "subclass_loss": subclass_loss_sum / total_samples,
        "class_acc": class_correct / total_samples,
        "subclass_acc_true_class": subclass_correct_true_class / total_samples,
        "joint_acc": joint_correct / total_samples
    }

In [ ]:
@torch.no_grad()
def validate_one_epoch(
    model,
    val_loader,
    device,
    use_amp=True
):
    model.eval()

    total_loss_sum = 0.0
    class_loss_sum = 0.0
    subclass_loss_sum = 0.0

    class_correct = 0
    subclass_correct_true_class = 0
    joint_correct = 0
    total_samples = 0

    progress_bar = tqdm(val_loader, desc="Validation", leave=False)

    for images, class_labels, subclass_labels in progress_bar:
        images = images.to(device)
        class_labels = class_labels.to(device).long()
        subclass_labels = subclass_labels.to(device).long()

        with torch.cuda.amp.autocast(enabled=use_amp):
            class_logits, subclass_logits_list = model(images)

            total_loss, class_loss, subclass_loss = hierarchical_loss(
                class_logits=class_logits,
                subclass_logits_list=subclass_logits_list,
                class_labels=class_labels,
                subclass_labels=subclass_labels,
                class_weight=CLASS_WEIGHT,
                subclass_weight=SUBCLASS_WEIGHT
            )

        batch_size = images.size(0)

        total_loss_sum += total_loss.item() * batch_size
        class_loss_sum += class_loss.item() * batch_size
        subclass_loss_sum += subclass_loss.item() * batch_size

        metrics = calculate_batch_metrics(
            class_logits=class_logits,
            subclass_logits_list=subclass_logits_list,
            class_labels=class_labels,
            subclass_labels=subclass_labels
        )

        class_correct += metrics["class_correct"]
        subclass_correct_true_class += metrics["subclass_correct_true_class"]
        joint_correct += metrics["joint_correct"]
        total_samples += metrics["total"]

        progress_bar.set_postfix({
            "loss": total_loss_sum / total_samples,
            "class_acc": class_correct / total_samples,
            "sub_acc": subclass_correct_true_class / total_samples,
            "joint_acc": joint_correct / total_samples
        })

    return {
        "loss": total_loss_sum / total_samples,
        "class_loss": class_loss_sum / total_samples,
        "subclass_loss": subclass_loss_sum / total_samples,
        "class_acc": class_correct / total_samples,
        "subclass_acc_true_class": subclass_correct_true_class / total_samples,
        "joint_acc": joint_correct / total_samples
    }

In [ ]:
def subclass_accuracy_by_true_class(subclass_logits_list, class_labels, subclass_labels):
    correct = 0
    total = class_labels.size(0)

    for class_idx, subclass_logits in enumerate(subclass_logits_list):
        mask = class_labels == class_idx

        if mask.any():
            preds = subclass_logits[mask].argmax(dim=1)
            correct += (preds == subclass_labels[mask]).sum().item()

    return correct, total

def joint_accuracy(class_logits, subclass_logits_list, class_labels, subclass_labels):
    class_preds = class_logits.argmax(dim=1)

    correct = 0
    total = class_labels.size(0)

    for i in range(total):
        pred_class = class_preds[i].item()
        pred_subclass = subclass_logits_list[pred_class][i].argmax().item()

        if pred_class == class_labels[i].item() and pred_subclass == subclass_labels[i].item():
            correct += 1

    return correct, total

In [ ]:
best_joint_acc = 0.0
best_epoch = 0

history = {
    "train_loss": [],
    "val_loss": [],
    "train_class_acc": [],
    "val_class_acc": [],
    "train_subclass_acc_true_class": [],
    "val_subclass_acc_true_class": [],
    "train_joint_acc": [],
    "val_joint_acc": []
}


# =========================
# Начинаем с обучения только голов
# =========================

if WARMUP_EPOCHS > 0:
    set_trainable_heads_only(model)
else:
    set_trainable_last_blocks(model, n_last_blocks=3)

optimizer = build_optimizer(model)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS
)


for epoch in range(1, NUM_EPOCHS + 1):

    # После warmup размораживаем последние блоки EfficientNet-B3
    if epoch == WARMUP_EPOCHS + 1 and WARMUP_EPOCHS > 0:
        print("\nРазмораживаем последние блоки EfficientNet-B3")

        set_trainable_last_blocks(model, n_last_blocks=3)

        optimizer = build_optimizer(model)

        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=NUM_EPOCHS - WARMUP_EPOCHS
        )

    print(f"\nEpoch [{epoch}/{NUM_EPOCHS}]")

    train_metrics = train_one_epoch(
        model=model,
        train_loader=train_loader,
        optimizer=optimizer,
        device=device,
        scaler=scaler,
        use_amp=use_amp
    )

    val_metrics = validate_one_epoch(
        model=model,
        val_loader=val_loader,
        device=device,
        use_amp=use_amp
    )

    scheduler.step()

    history["train_loss"].append(train_metrics["loss"])
    history["val_loss"].append(val_metrics["loss"])

    history["train_class_acc"].append(train_metrics["class_acc"])
    history["val_class_acc"].append(val_metrics["class_acc"])

    history["train_subclass_acc_true_class"].append(
        train_metrics["subclass_acc_true_class"]
    )
    history["val_subclass_acc_true_class"].append(
        val_metrics["subclass_acc_true_class"]
    )

    history["train_joint_acc"].append(train_metrics["joint_acc"])
    history["val_joint_acc"].append(val_metrics["joint_acc"])

    print(
        f"Train Loss: {train_metrics['loss']:.4f} | "
        f"Class Loss: {train_metrics['class_loss']:.4f} | "
        f"Subclass Loss: {train_metrics['subclass_loss']:.4f}"
    )

    print(
        f"Train Class Acc: {train_metrics['class_acc']:.4f} | "
        f"Train Subclass Acc: {train_metrics['subclass_acc_true_class']:.4f} | "
        f"Train Joint Acc: {train_metrics['joint_acc']:.4f}"
    )

    print(
        f"Val Loss: {val_metrics['loss']:.4f} | "
        f"Class Loss: {val_metrics['class_loss']:.4f} | "
        f"Subclass Loss: {val_metrics['subclass_loss']:.4f}"
    )

    print(
        f"Val Class Acc: {val_metrics['class_acc']:.4f} | "
        f"Val Subclass Acc: {val_metrics['subclass_acc_true_class']:.4f} | "
        f"Val Joint Acc: {val_metrics['joint_acc']:.4f}"
    )

    # Сохраняем лучшую модель по joint accuracy
    if val_metrics["joint_acc"] > best_joint_acc:
        best_joint_acc = val_metrics["joint_acc"]
        best_epoch = epoch

        checkpoint = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_joint_acc": best_joint_acc,
            "val_class_acc": val_metrics["class_acc"],
            "val_subclass_acc_true_class": val_metrics["subclass_acc_true_class"],
            "val_joint_acc": val_metrics["joint_acc"],
            "history": history
        }

        torch.save(checkpoint, BEST_MODEL_PATH)

        print(
            f"Сохранена лучшая модель: epoch={best_epoch}, "
            f"val_joint_acc={best_joint_acc:.4f}"
        )


# Сохраняем последнюю модель тоже
last_checkpoint = {
    "epoch": NUM_EPOCHS,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "best_epoch": best_epoch,
    "best_joint_acc": best_joint_acc,
    "history": history
}

torch.save(last_checkpoint, LAST_MODEL_PATH)

print("\nОбучение завершено.")
print(f"Лучшая эпоха: {best_epoch}")
print(f"Лучший Val Joint Acc: {best_joint_acc:.4f}")
print(f"Лучшая модель сохранена в: {BEST_MODEL_PATH}")
print(f"Последняя модель сохранена в: {LAST_MODEL_PATH}")